# Analisis Pemilihan Model (Manual)
**Mata Kuliah:** Machine Learning (Kelas C), Bapak Adi Purnawan

**Anggota:** Deliana Br Manalu (2305551036) · Ravi Arnan Irianto (2305551076) · Ezza Putra Wibawa (2305551144) · Devin (2305551173)

---

Notebook ini membandingkan dua algoritma klasifikasi yang diimplementasikan **manual**
(tanpa scikit-learn): **K-Nearest Neighbors (KNN)** dan **Naive Bayes**.

Hasilnya dievaluasi dengan metrik yang dihitung manual (akurasi, presisi, recall,
F1-score, confusion matrix). Library hanya dipakai untuk manipulasi data (pandas)
dan perhitungan numeric (numpy). Pemodelan library akan dilakukan minggu depan.

Dataset yang dipakai: hasil pembersihan dari notebook 01
(`artifacts/stroke_pemeriksaan_awal.csv`). Hanya kolom numerik yang dipakai:
age, hypertension, heart_disease, avg_glucose_level, bmi. Target: stroke.

## 1. Muat Dataset

In [ ]:
import pandas as pd
import numpy as np
import math
import random

random.seed(42)
np.random.seed(42)

# Bisa juga pakai file lokal: ../data/healthcare-stroke-data.csv
URL_DATA = "https://raw.githubusercontent.com/ravi-arnan/machine-learning/main/data/healthcare-stroke-data.csv"
df = pd.read_csv(URL_DATA)

# Hanya pakai kolom numerik dan isi BMI kosong dengan median
df = df[["age", "hypertension", "heart_disease", "avg_glucose_level", "bmi", "stroke"]].copy()
df["bmi"] = df["bmi"].fillna(df["bmi"].median())

print("Ukuran dataset:", df.shape)
print("5 baris pertama:")
df.head()

## 2. Train-Test Split (Manual)

Split 80:20 tanpa library. Acak indeks lalu potong.

In [ ]:
def train_test_split_manual(data, target_col, test_size=0.2, seed=42):
    """Split data menjadi train dan test secara manual."""
    random.seed(seed)
    n = len(data)
    n_test = int(n * test_size)

    indices = list(range(n))
    random.shuffle(indices)

    test_idx = indices[:n_test]
    train_idx = indices[n_test:]

    X_train = data.iloc[train_idx].drop(columns=[target_col]).values
    y_train = data.iloc[train_idx][target_col].values
    X_test = data.iloc[test_idx].drop(columns=[target_col]).values
    y_test = data.iloc[test_idx][target_col].values

    return X_train, X_test, y_train, y_test


X_train, X_test, y_train, y_test = train_test_split_manual(
    df, target_col="stroke", test_size=0.2, seed=42
)

print(f"Train: {X_train.shape[0]} baris")
print(f"Test:  {X_test.shape[0]} baris")
print(f"Proporsi stroke di train: {y_train.mean():.3f}")
print(f"Proporsi stroke di test:  {y_test.mean():.3f}")

## 3. Fungsi Evaluasi (Manual)

Semua metrik dihitung dari confusion matrix tanpa library.

In [ ]:
def confusion_matrix_manual(y_true, y_pred):
    """Hitung TP, TN, FP, FN manual."""
    tp = tn = fp = fn = 0
    for t, p in zip(y_true, y_pred):
        if t == 1 and p == 1:
            tp += 1
        elif t == 0 and p == 0:
            tn += 1
        elif t == 0 and p == 1:
            fp += 1
        elif t == 1 and p == 0:
            fn += 1
    return tp, tn, fp, fn


def hitung_metrik(y_true, y_pred):
    """Hitung akurasi, presisi, recall, F1 manual."""
    tp, tn, fp, fn = confusion_matrix_manual(y_true, y_pred)

    akurasi = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
    presisi = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * presisi * recall / (presisi + recall) if (presisi + recall) > 0 else 0

    return {
        "akurasi": round(akurasi, 4),
        "presisi": round(presisi, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }

## 4. Model 1: K-Nearest Neighbors (KNN) Manual

Hitung jarak Euclidean ke semua data train, ambil K tetangga terdekat, lalu voting.

In [ ]:
def euclidean_distance(a, b):
    """Jarak Euclidean antara dua vektor."""
    return math.sqrt(sum((x - y) ** 2 for x, y in zip(a, b)))


def knn_predict(X_train, y_train, X_test, k=5):
    """KNN: prediksi satu titik uji."""
    y_pred = []

    for titik_uji in X_test:
        jarak = []
        for i, titik_latih in enumerate(X_train):
            d = euclidean_distance(titik_uji, titik_latih)
            jarak.append((d, y_train[i]))

        jarak.sort(key=lambda x: x[0])
        tetangga = jarak[:k]

        voting = 0
        for _, label in tetangga:
            voting += label

        pred = 1 if voting > k / 2 else 0
        y_pred.append(pred)

    return np.array(y_pred)


print("Menjalankan KNN (k=5)...")
y_pred_knn = knn_predict(X_train, y_train, X_test, k=5)
print("Selesai.")

In [ ]:
metrik_knn = hitung_metrik(y_test, y_pred_knn)

print("=" * 50)
print("HASIL KNN (k=5)")
print("=" * 50)
print(f"  Akurasi:  {metrik_knn['akurasi']}")
print(f"  Presisi:  {metrik_knn['presisi']}")
print(f"  Recall:   {metrik_knn['recall']}")
print(f"  F1-score: {metrik_knn['f1']}")
print()
print("Confusion Matrix:")
print(f"  TP={metrik_knn['tp']}  FP={metrik_knn['fp']}")
print(f"  FN={metrik_knn['fn']}  TN={metrik_knn['tn']}")

### 4a. Coba Beberapa Nilai K

Cari nilai K yang paling optimal dengan mencoba beberapa angka.

In [ ]:
print(f"{'K':>3} {'Akurasi':>8} {'Presisi':>8} {'Recall':>8} {'F1':>8}")
print("-" * 40)

for k in [1, 3, 5, 7, 9, 11, 15, 21]:
    pred = knn_predict(X_train, y_train, X_test, k=k)
    m = hitung_metrik(y_test, pred)
    print(f"{k:>3} {m['akurasi']:>8} {m['presisi']:>8} {m['recall']:>8} {m['f1']:>8}")

## 5. Model 2: Naive Bayes (Gaussian) Manual

Hitung mean dan std tiap fitur per kelas, lalu hitung probabilitas dengan rumus
Gaussian PDF. Pilih kelas dengan probabilitas tertinggi.

In [ ]:
def gaussian_pdf(x, mean, std):
    """Hitung probabilitas Gaussian."""
    if std == 0:
        return 1.0 if x == mean else 0.0001
    eksponen = math.exp(-((x - mean) ** 2) / (2 * std ** 2))
    return (1 / (math.sqrt(2 * math.pi) * std)) * eksponen


class NaiveBayesManual:
    """Gaussian Naive Bayes buatan sendiri."""

    def fit(self, X, y):
        n, m = X.shape
        self.kelas = sorted(set(y))
        self.prior = {}
        self.mean = {}
        self.std = {}

        for c in self.kelas:
            Xc = X[y == c]
            self.prior[c] = len(Xc) / n
            self.mean[c] = Xc.mean(axis=0)
            self.std[c] = Xc.std(axis=0)

    def predict(self, X):
        y_pred = []
        for titik in X:
            prob_kelas = {}
            for c in self.kelas:
                prob = math.log(self.prior[c])
                for i in range(len(titik)):
                    prob += math.log(gaussian_pdf(titik[i], self.mean[c][i], self.std[c][i]) + 1e-10)
                prob_kelas[c] = prob
            y_pred.append(max(prob_kelas, key=prob_kelas.get))
        return np.array(y_pred)


nb = NaiveBayesManual()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)
print("Naive Bayes selesai.")

In [ ]:
metrik_nb = hitung_metrik(y_test, y_pred_nb)

print("=" * 50)
print("HASIL NAIVE BAYES")
print("=" * 50)
print(f"  Akurasi:  {metrik_nb['akurasi']}")
print(f"  Presisi:  {metrik_nb['presisi']}")
print(f"  Recall:   {metrik_nb['recall']}")
print(f"  F1-score: {metrik_nb['f1']}")
print()
print("Confusion Matrix:")
print(f"  TP={metrik_nb['tp']}  FP={metrik_nb['fp']}")
print(f"  FN={metrik_nb['fn']}  TN={metrik_nb['tn']}")

## 6. Perbandingan Kedua Model

In [ ]:
print("=" * 55)
print("PERBANDINGAN MODEL")
print("=" * 55)
print(f"{'Metrik':<10} {'KNN (k=5)':<15} {'Naive Bayes':<15}")
print("-" * 40)

for metrik in ["akurasi", "presisi", "recall", "f1"]:
    knn_val = metrik_knn[metrik]
    nb_val = metrik_nb[metrik]
    print(f"{metrik:<10} {knn_val:<15} {nb_val:<15}")

print()
if metrik_knn["f1"] > metrik_nb["f1"]:
    print("Kesimpulan: KNN (k=5) lebih baik berdasarkan F1-score.")
elif metrik_nb["f1"] > metrik_knn["f1"]:
    print("Kesimpulan: Naive Bayes lebih baik berdasarkan F1-score.")
else:
    print("Kesimpulan: Kedua model memiliki performa yang setara.")

## 7. Catatan untuk Minggu Depan

Implementasi manual ini menunjukkan cara kerja algoritma dari dalam. Minggu depan
kita akan pakai scikit-learn untuk model yang lebih optimal (dengan tuning
hyperparameter, scaling fitur, dsb) dan algoritma lain seperti Logistic
Regression, Decision Tree, Random Forest, SVM.